# EPIC Clarity Device Exposure Hydration

This notebook hydrates the OMOP DEVICE_EXPOSURE table from EPIC Clarity OR implant data.

## Source Table
- `_exponent._bronze_epic_clarity_*.dbo_OR_IMP`

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Transform EPIC device/implant data
%sql
CREATE OR REPLACE TEMP VIEW device_exposure_silver AS
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'OR_IMP', 'IMPLANT_ID', oi.IMPLANT_ID) AS device_source_value,
    oi.IMPLANT_NAME AS device_name,
    oi.STATUS_C_NAME AS device_type_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity_prod01_vw.dbo_OR_IMP oi
WHERE oi.IMPLANT_ID IS NOT NULL

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.device_exposure AS target
USING device_exposure_silver AS source
ON target.device_exposure_source_value = source.device_exposure_source_value

WHEN MATCHED AND NOT (
    target.device_source_value <=> source.device_source_value
)
THEN UPDATE SET
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    device_exposure_source_value,
    updated_tsp
)
VALUES (
    source.device_exposure_source_value,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_device_exposure (
    source_system,
    device_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.device_exposure_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT device_exposure_source_value, updated_tsp
    FROM _exponent.omop_silver.device_exposure
    WHERE device_exposure_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_device_exposure x
    ON s.device_exposure_source_value = x.device_exposure_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with mapping to get device_exposure_id
%sql
CREATE OR REPLACE TEMP VIEW device_exposure_gold AS
SELECT
    m.device_exposure_id,
    s.updated_tsp
FROM _exponent.omop_silver.device_exposure s
INNER JOIN _exponent.omop_mapping.source_to_device_exposure m
    ON s.device_exposure_source_value = m.device_exposure_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.device_exposure AS target
USING device_exposure_gold AS source
ON target.device_exposure_id = source.device_exposure_id

WHEN NOT MATCHED THEN INSERT (
    device_exposure_id
)
VALUES (
    source.device_exposure_id
)